In [2]:
# Install the required system dependency
!apt-get update -qq
!apt-get install -y -qq zstd curl

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import os
import subprocess
import time

# Avoid starting a second server if one is already running
server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

!curl -s http://127.0.0.1:11434/api/tags

{"models":[]}

In [5]:
!ollama pull qwen3:8b
!ollama pull qwen3:latest
!ollama list
!echo $OLLAMA_MODELS
!ollama ps

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import subprocess
import time
import os

# Create a permanent folder in  Drive
os.makedirs('/content/drive/MyDrive/ollama_models', exist_ok=True)

# Set the environment variable  Ollama saves to Drive
os.environ['OLLAMA_MODELS'] = '/content/drive/MyDrive/ollama_models'

# Start the server in the background
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

In [12]:
# 1. Point Ollama to  Google Drive folder
%env OLLAMA_MODELS=/content/drive/MyDrive/ollama_models

# 2. Start the server in the background silently
!nohup ollama serve > ollama.log 2>&1 &

env: OLLAMA_MODELS=/content/drive/MyDrive/ollama_models


In [ ]:
!ollama run qwen3:latest "What is the capital of India?"

Thinking...
Okay, the user is asking for the capital of India. I need to make sure I pr
provide the correct answer. Let me think... I know that India is a country 
in South Asia, and its capital is a well-known city. Wait, isn't it New Del
Delhi? Yes, I'm pretty sure that's right. But let me double-check to avoid 
any mistakes. Sometimes people confuse it with other cities like Mumbai or 
Kolkata, which are major cities but not the capital. New Delhi is the capit
capital, and it's also the seat of the Government of India. The Indian Parl
Parliament is located there as well. I think that's correct. I should also 
mention that New Delhi is part of the National Capital Territory, which is 
a union territory. That might add some context. Alright, I'm confident the 
answer is New Delhi.
...done thinking.

The capital of India is **New Delhi**. It is also the seat of the Governmen
Government of India and home to the Parliament of India. New Delhi is part 
of the **National Capital Territory 

In [14]:
!pip install biopython GEOparse openai ollama pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.2 MB/s eta 0:00:00


In [16]:
import os
import re
import sys
import json
import time
import logging
from datetime import datetime
from typing import List, Optional

import requests
import pandas as pd
from pydantic import BaseModel, Field
from Bio import Entrez

try:
    import ollama
    HAS_OLLAMA = True
except ImportError:
    HAS_OLLAMA = False
try:
    import GEOparse
    HAS_GEOPARSE = True
except ImportError:
    HAS_GEOPARSE = False

# --- CONFIGURATION  ---
PROVIDER = "ollama"
OLLAMA_MODEL = "qwen3:latest"
EMAIL = "naga7dna@gamil.com"

MAX_RESULTS = 10
SCORE_THRESHOLD = 6
TOP_K_EXTRACT = 2
FETCH_BATCH = 2
LLM_TEMP = 0.0
SCORE_BATCH = 1
USE_PUBTATOR = True
USE_GEO = True
GEO_MAX_SERIES_PER_PAPER = 1
GEO_DATASET_HITS = 1

PT_API = "https://www.ncbi.nlm.nih.gov/research/pubtator3-api"

# ---------------------------------------------------------------- LOGGING ---
def setup_logging() -> str:
    log_file = f"pipeline_{datetime.now():%Y%m%d_%H%M%S}.log"
    fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s")
    root = logging.getLogger()
    root.setLevel(logging.INFO)
    fh = logging.FileHandler(log_file, encoding="utf-8")
    fh.setFormatter(fmt)
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    root.addHandler(fh)
    root.addHandler(sh)
    logging.getLogger("httpx").setLevel(logging.WARNING)
    return log_file

log = logging.getLogger("pipeline")

# ------------------------------------------------- LLM OUTPUT SCHEMAS -------
class SearchPlan(BaseModel):
    intent_summary: str = ""
    detected_pmids: List[str] = Field(default_factory=list)
    disease_focus: str = ""
    comparison_groups: str = ""
    omics_methods: str = ""
    year_from: Optional[int] = None
    pubmed_query: str = ""

class Verdict(BaseModel):
    pmid: str
    score: int
    reason: str = ""

class VerdictList(BaseModel):
    verdicts: List[Verdict] = Field(default_factory=list)

class Extraction(BaseModel):
    study_type: str = ""
    datasets_mentioned: List[str] = Field(default_factory=list)
    accessions: List[str] = Field(default_factory=list)
    experimental_conditions: List[str] = Field(default_factory=list)
    n_control: Optional[int] = None
    n_metastasis: Optional[int] = None
    n_total: Optional[int] = None
    evidence_quote: str = ""
    extraction_confidence: str = ""

class CharLabel(BaseModel):
    text: str = ""
    group: str = ""

class CharLabelList(BaseModel):
    labels: List[CharLabel] = Field(default_factory=list)

def _schema_json(schema):
    try:
        return schema.model_json_schema()
    except AttributeError:
        return schema.schema()

def _validate(schema, text):
    try:
        return schema.model_validate_json(text)
    except AttributeError:
        return schema.parse_raw(text)

def _parse_json(text, schema):
    text = (text or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        return _validate(schema, text)
    except Exception:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m:
            return _validate(schema, m.group(0))
        raise

def _call_ollama(messages, schema):
    try:
        resp = ollama.chat(model=OLLAMA_MODEL, messages=messages,
                           format=_schema_json(schema),
                           options={"temperature": LLM_TEMP})
    except Exception:
        resp = ollama.chat(model=OLLAMA_MODEL, messages=messages,
                           format="json", options={"temperature": LLM_TEMP})
    return resp["message"]["content"]

def llm(messages, schema):
    if not HAS_OLLAMA:
        raise RuntimeError("Ollama is not installed. Run !pip install ollama")

    for attempt in range(1, 4):
        t = time.time()
        try:
            content = _call_ollama(messages, schema)
            obj = _parse_json(content, schema)
            log.info(f"[ollama] {schema.__name__} OK ({time.time()-t:.1f}s)")
            return obj
        except Exception as e:
            wait = 3 * attempt
            log.warning(f"[ollama] attempt {attempt}/3 failed (retry in {wait}s): {str(e)[:150]}")
            time.sleep(wait)
    raise RuntimeError("LLM failed on all attempts.")

# --------------------------------------------------------- PUBTATOR3 --------
def pubtator_search(text: str, size: int = 100) -> List[str]:
    try:
        r = requests.get(f"{PT_API}/search/", params={"text": text, "size": size}, timeout=60)
        r.raise_for_status()
        js = r.json()
        pmids = js.get("pmids") if isinstance(js, dict) else []
        if not pmids and isinstance(js, dict) and isinstance(js.get("results"), list):
            pmids = [x.get("pmid") for x in js["results"] if x.get("pmid")]
        return [str(p) for p in (pmids or [])][:size]
    except Exception as e:
        return []

def pubtator_annotate(pmids: List[str]) -> dict:
    out = {}
    for i in range(0, len(pmids), 100):
        chunk = ",".join(p for p in pmids[i:i+100] if p)
        if not chunk: continue
        try:
            url = f"{PT_API}/publications/export/pubtator?pmids={chunk}&concepts=gene,disease,cellline"
            r = requests.get(url, timeout=120)
            r.raise_for_status()
            for line in r.text.splitlines():
                f = line.split("\t")
                if len(f) != 7: continue
                pmid, _sec, etype, _id, _s, _e, text = f
                d = out.setdefault(pmid, {"genes": set(), "diseases": set(), "celllines": set()})
                if etype == "Gene": d["genes"].add(text)
                elif etype == "Disease": d["diseases"].add(text)
                elif etype == "CellLine": d["celllines"].add(text)
            time.sleep(1)
        except Exception as e:
            pass
    return out

def add_pubtator_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    ann = pubtator_annotate(df["PMID"].tolist())
    df["Genes"] = df["PMID"].map(lambda p: "; ".join(sorted(ann.get(p, {}).get("genes", []))[:15]))
    df["Diseases"] = df["PMID"].map(lambda p: "; ".join(sorted(ann.get(p, {}).get("diseases", []))[:10]))
    return df

# ------------------------------------------------------------- GEO ----------
_geo_cache = {}

def _kw_classify(ch: str) -> str:
    c = ch.lower()
    if "metast" in c: return "metastasis"
    if any(k in c for k in ("normal", "adjacent", "control", "non-tumor", "healthy")): return "control"
    if any(k in c for k in ("tumor", "tumour", "carcinoma", "cancer")): return "tumor"
    return "other"

def geo_group_counts(gse_id: str):
    if not HAS_GEOPARSE: return None
    gse_id = gse_id.upper().strip()
    if gse_id in _geo_cache: return _geo_cache[gse_id]
    try:
        g = GEOparse.get_GEO(geo=gse_id, destdir="geo_cache", silent=True)
    except Exception as e:
        _geo_cache[gse_id] = None
        return None

    sample_chars = {}
    for name, gsm in g.gsms.items():
        raw = gsm.metadata.get("characteristics_ch1") or gsm.metadata.get("title") or []
        chars = re.sub(r"\s+", " ", " | ".join(str(x) for x in raw)).strip()
        sample_chars[name] = chars
    unique = sorted({c for c in sample_chars.values() if c})
    if not unique:
        _geo_cache[gse_id] = None
        return None

    labels = {c: _kw_classify(c) for c in unique}

    counts = {"control": 0, "metastasis": 0, "tumor": 0, "other": 0}
    for c in sample_chars.values():
        counts[labels.get(c, "other")] += 1
    res = {"n_total": len(sample_chars), "n_control": counts["control"],
           "n_metastasis": counts["metastasis"], "n_tumor": counts["tumor"],
           "n_other": counts["other"]}
    _geo_cache[gse_id] = res
    return res

def geo_search_datasets(disease: str, extra: str = "", n: int = 15) -> pd.DataFrame:
    if not disease: return pd.DataFrame()
    try:
        term = f'"{disease}"[All Fields]'
        if extra: term += f' AND ({extra})'
        h = Entrez.esearch(db="gds", term=term, retmax=n * 3)
        ids = Entrez.read(h)["IdList"]; h.close()
        if not ids: return pd.DataFrame()
        h = Entrez.esummary(db="gds", id=",".join(ids))
        rows = []
        for s in Entrez.read(h):
            if str(s.get("entrytype", "")).upper() != "GSE": continue
            pms = s.get("PubMedIds") or s.get("PubmedIds") or []
            rows.append({
                "Accession": str(s.get("Accession", "")),
                "Title": str(s.get("title", "")),
                "Summary": str(s.get("summary", ""))[:200],
                "N_Samples": s.get("n_samples", ""),
                "Organism": str(s.get("taxon", "")),
                "PMIDs": ";".join(str(x) for x in pms),
            })
            if len(rows) >= n: break
        h.close()
        return pd.DataFrame(rows)
    except Exception as e:
        return pd.DataFrame()

def geo_cross_check(result: pd.DataFrame) -> pd.DataFrame:
    if not HAS_GEOPARSE: return result
    result = result.copy()
    tot, ctl, met, src = [], [], [], []
    for _, row in result.iterrows():
        accs = [a.strip().upper() for a in re.split(r"[;,]", str(row.get("Accessions") or "")) if a.strip()]
        gses = [a for a in accs if a.startswith("GSE")][:GEO_MAX_SERIES_PER_PAPER]
        got, used = None, ""
        for gse in gses:
            got = geo_group_counts(gse)
            if got:
                used = gse
                break
        if got:
            tot.append(got["n_total"]); ctl.append(got["n_control"]); met.append(got["n_metastasis"])
        else:
            tot.append(None); ctl.append(None); met.append(None)
        src.append(used or "-")
    result["GEO_Series"] = src
    result["GEO_N_Total"] = tot
    result["GEO_N_Control"] = ctl
    result["GEO_N_Metastasis"] = met
    return result

def finalize_sample_counts(result: pd.DataFrame) -> pd.DataFrame:
    result = result.copy()
    for col in ("N_Control", "N_Metastasis", "N_Total", "GEO_N_Total", "GEO_N_Control", "GEO_N_Metastasis"):
        if col in result.columns:
            result[col] = pd.to_numeric(result[col], errors="coerce")
    for base, fin in (("N_Control", "N_Control_Final"), ("N_Metastasis", "N_Metastasis_Final"), ("N_Total", "N_Total_Final")):
        geo_col = "GEO_" + base
        if geo_col in result.columns:
            result[fin] = result[geo_col].combine_first(result[base])
        else:
            result[fin] = result[base]
    return result

# ------------------------------------------------------------ STAGES --------

STAGE1_SYSTEM = """You convert a researcher's free-form request into simple PubMed search terms. Return JSON with:
- intent_summary: short description of what the user wants.
- disease_focus: main disease or topic.
- pubmed_query: A clean, simple search query containing ONLY keywords separated by AND (e.g., mRNA AND vaccines AND lung AND cancer). DO NOT use bracketed tags like [Title] or [tiab]."""

def stage1_plan(user_input: str) -> SearchPlan:
    log.info("[Stage 1] Interpreting request with LLM ...")
    return llm([{"role": "system", "content": STAGE1_SYSTEM},
                {"role": "user", "content": user_input}], SearchPlan)

def clean_pubmed_query(q: str) -> str:
    """Strips markdown and hallucinated PubMed bracket fields that break Entrez"""
    if not q: return ""
    q = re.sub(r"^```(?:json)?\s*", "", q)
    q = re.sub(r"\s*```$", "", q)
    q = re.sub(r"\[[a-zA-Z]+\]", "", q) # Strips [SO], [dp], [tiab] etc.
    q = q.strip().strip('"').strip("'")
    return q

def generate_fallback_query(text: str) -> str:
    """Converts conversational sentences into strict PubMed boolean logic"""
    stop_words = {"find", "recent", "papers", "on", "and", "for", "about", "treatments", "research",
                  "studies", "with", "the", "a", "an", "of", "in", "to", "how", "what", "is", "are",
                  "does", "show", "me", "articles"}

    clean_text = re.sub(r"[^\w\s]", " ", text).lower()
    keywords = [w for w in clean_text.split() if w not in stop_words and len(w) > 2]
    return " AND ".join(keywords)

def stage2_search(query: str, max_results: int) -> List[str]:
    try:
        h = Entrez.esearch(db="pubmed", term=query, retmax=max_results, sort="relevance")
        record = Entrez.read(h)
        h.close()
        return [str(i) for i in record.get("IdList", [])]
    except Exception as e:
        log.warning(f"PubMed search failed for query '{query}': {e}")
        return []

def stage2_fetch(id_list: List[str]) -> pd.DataFrame:
    rows = []
    for chunk in [id_list[i:i + FETCH_BATCH] for i in range(0, len(id_list), FETCH_BATCH)]:
        try:
            h = Entrez.efetch(db="pubmed", id=",".join(chunk), retmode="xml")
            xml = Entrez.read(h)
            h.close()
            for pa in xml.get("PubmedArticle", []):
                cit, art = pa["MedlineCitation"], pa["MedlineCitation"]["Article"]
                pmid = str(cit["PMID"])
                title = str(art.get("ArticleTitle", "")).strip()
                at = art.get("Abstract", {}).get("AbstractText", [])
                abstract = " ".join([str(p) for p in (at if isinstance(at, list) else [at])])
                year = art.get("Journal", {}).get("JournalIssue", {}).get("PubDate", {}).get("Year", "Unknown")
                journal = str(art.get("Journal", {}).get("Title", ""))
                rows.append({"PMID": pmid, "Title": title, "Abstract": abstract, "Year": year, "Journal": journal, "URL": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"})
        except Exception:
            pass
    return pd.DataFrame(rows)

STAGE3A_SYSTEM = """You screen scientific papers for a researcher. Score each 0-10 for relevance to the RESEARCH NEED. Return a verdict for EVERY paper using the exact PMIDs given."""

def stage3a_filter(df: pd.DataFrame, user_need: str) -> pd.DataFrame:
    verdicts = {}
    for i in range(0, len(df), SCORE_BATCH):
        batch = df.to_dict("records")[i:i + SCORE_BATCH]
        listing = "\n\n".join(f"PMID {p['PMID']}\nTITLE: {p['Title']}\nABSTRACT: {p['Abstract'][:1500]}" for p in batch)
        try:
            got = llm([{"role": "system", "content": STAGE3A_SYSTEM},
                       {"role": "user", "content": f"RESEARCH NEED: {user_need}\n\nPAPERS:\n{listing}"}], VerdictList).verdicts
            for v in got: verdicts[str(v.pmid)] = v
        except Exception:
            pass
    df = df.copy()
    df["Score"] = df["PMID"].map(lambda x: verdicts[x].score if x in verdicts else 0)
    df["Reason"] = df["PMID"].map(lambda x: verdicts[x].reason if x in verdicts else "scoring failed")
    return df.sort_values("Score", ascending=False).reset_index(drop=True)

ACCESSION_RE = re.compile(
    r"\b(?:GSE\d+|GDS\d+|SRP\d+|ERP\d+|PRJNA\d+|PRJEB\d+|SCP\d+|"
    r"E-MTAB-\d+|E-GEOD-\d+|phs\d{6}|TCGA|CPTAC|GTEx)\b", re.IGNORECASE)

STAGE3B_SYSTEM = """You extract structured metadata from ONE biomedical paper (title + abstract). Only extract facts EXPLICITLY present in the text."""

def stage3b_extract(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, p in df.iterrows():
        try:
            ex = llm([{"role": "system", "content": STAGE3B_SYSTEM},
                      {"role": "user", "content": f"TITLE: {p['Title']}\nABSTRACT: {p['Abstract'][:3500]}"}], Extraction)
        except Exception:
            ex = Extraction()
        regex_hits = [a.upper() for a in ACCESSION_RE.findall(f"{p['Abstract']} {p['Title']}")]
        rows.append({
            "PMID": p["PMID"], "Title": p["Title"], "Score": p["Score"], "Reason": p["Reason"],
            "Study_Type": ex.study_type, "Datasets": "; ".join(ex.datasets_mentioned),
            "Accessions": "; ".join(sorted(set(regex_hits) | {a.upper() for a in ex.accessions})),
            "N_Control": ex.n_control, "N_Metastasis": ex.n_metastasis, "N_Total": ex.n_total,
            "Evidence_Quote": ex.evidence_quote, "Confidence": ex.extraction_confidence,
            "Abstract": p["Abstract"], "URL": p["URL"],
        })
    return pd.DataFrame(rows)

def verify_quotes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    def ok(row):
        q = re.sub(r"\s+", " ", str(row.get("Evidence_Quote") or "")).strip()
        t = re.sub(r"\s+", " ", str(row.get("Abstract") or ""))
        return bool(q) and q[:120] in t
    df["Quote_Verified"] = df.apply(ok, axis=1)
    return df

# ------------------------------------------------------------- PIPELINE -----
def run_pipeline(user_input: str):
    t0 = time.time()
    log.info("=" * 70)
    log.info(f"RUN START | input: {user_input}")

    # ---- Stage 1
    plan = stage1_plan(user_input)
    llm_query = clean_pubmed_query(plan.pubmed_query)
    log.info(f"[Stage 1] Generated PubMed Query: '{llm_query}'")

    # ---- Stage 2 Primary Search
    pmids = []
    if llm_query:
        pmids = stage2_search(llm_query, MAX_RESULTS)

    # ---- Stage 2 Fallback
    if not pmids:
        fallback_query = generate_fallback_query(user_input)
        log.warning(f"[Stage 2] LLM query returned 0 results. Falling back to logical keyword search: '{fallback_query}'")
        pmids = stage2_search(fallback_query, MAX_RESULTS)

    if not pmids:
        log.error("No papers found even after fallback keyword search.")
        return

    log.info(f"[Stage 2] Retrieved {len(pmids)} PMIDs")

    df = stage2_fetch(pmids)
    if df.empty:
        log.error("Failed to fetch metadata from PubMed.")
        return

    if USE_PUBTATOR:
        df = add_pubtator_columns(df)

    # ---- Stage 3a
    df = stage3a_filter(df, user_input)
    df.to_csv("all_papers_scored.csv", index=False)

    relevant = df[df["Score"] >= SCORE_THRESHOLD]
    if relevant.empty:
        log.error("No paper passed the threshold. Lower SCORE_THRESHOLD or rephrase.")
        return

    # ---- Stage 3b
    result = stage3b_extract(relevant.head(TOP_K_EXTRACT))

    if USE_GEO:
        result = geo_cross_check(result)
        geo_df = geo_search_datasets(plan.disease_focus, plan.omics_methods, GEO_DATASET_HITS)
        if not geo_df.empty:
            geo_df.to_csv("geo_datasets.csv", index=False)

    result = finalize_sample_counts(result)
    result = verify_quotes(result)
    result.to_csv("extracted_papers.csv", index=False)

    # ---- preview
    cols = ["PMID", "Score", "Title", "Datasets", "Accessions", "N_Total_Final", "Confidence"]
    cols = [c for c in cols if c in result.columns]
    pv = result[cols].copy()
    if "Title" in pv:
        pv["Title"] = pv["Title"].astype(str).str.slice(0, 55) + "..."

    print("\n" + "=" * 25 + f" TOP {len(result)} EXTRACTED PAPERS " + "=" * 25)
    print(pv.to_string(index=False, max_colwidth=25))
    log.info(f"RUN FINISHED in {time.time()-t0:.0f}s")


if __name__ == "__main__":
    setup_logging()
    Entrez.email = EMAIL

    RESEARCH_QUERY = "endometrial cancer is the most common type of cancer in the female genital tract"
    run_pipeline(RESEARCH_QUERY)

INFO:pipeline:======================================================================


2026-10-08 14:06:58,501 | INFO    | ======================================================================
2026-10-08 14:06:58,501 | INFO    | ======================================================================


INFO:pipeline:RUN START | input: endometrial cancer is the most common type of cancer in the female genital tract


2026-10-08 14:06:58,503 | INFO    | RUN START | input: endometrial cancer is the most common type of cancer in the female genital tract
2026-10-08 14:06:58,503 | INFO    | RUN START | input: endometrial cancer is the most common type of cancer in the female genital tract


INFO:pipeline:[Stage 1] Interpreting request with LLM ...


2026-10-08 14:06:58,505 | INFO    | [Stage 1] Interpreting request with LLM ...
2026-10-08 14:06:58,505 | INFO    | [Stage 1] Interpreting request with LLM ...


2026-10-08 14:06:58,512 | WARNING | [ollama] attempt 1/3 failed (retry in 3s): Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
2026-10-08 14:06:58,512 | WARNING | [ollama] attempt 1/3 failed (retry in 3s): Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


2026-10-08 14:07:01,517 | WARNING | [ollama] attempt 2/3 failed (retry in 6s): Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
2026-10-08 14:07:01,517 | WARNING | [ollama] attempt 2/3 failed (retry in 6s): Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


2026-10-08 14:07:07,521 | WARNING | [ollama] attempt 3/3 failed (retry in 9s): Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download
2026-10-08 14:07:07,521 | WARNING | [ollama] attempt 3/3 failed (retry in 9s): Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


RuntimeError: LLM failed on all attempts.